# Workflow for PDMS Molecular Dynamics Simulations Using LAMMPS

This notebook demonstrates a workflow for building, simulating, and analysing **polydimethylsiloxane (PDMS)** systems using LAMMPS.

### 1. Polymer Construction

- PDMS chains are generated using **SwiftPol**:  
  https://github.com/matta-research-group/SwiftPol

- The generated chains are converted into:
  - OpenFF `Molecule` --> `Topology` --> `Interchange` objects

- Force field parameters are assigned using a custom `bespoke_ff.offxml` force field generated with **Presto**:  
  https://github.com/cole-group/presto

### 2. Initial Configuration Generation

- **Polyply** is used to generate the initial coordiantes for the box which are updated in the topology and made into an interchange \
  https://github.com/marrink-lab/polyply_1.0

### 3. Molecular Dynamics Simulations

LAMMPS is used as the simulation engine (although the workflow could be adapted to use **GROMACS** or **OpenMM**).

The simulation protocol consists of:

1. **Energy minimisation**
2. **Heating** to **600 K**
3. **Cooling** to **300 K**
4. **NPT MD** at **300 K** **10 ns**

Heating above the PDMS glass transition temperature promotes equilibration before slow cooling.

### 4. Output Files

`.lammpstrj` trajectory file containing xyz coordinates \
`.thermo` thermo dynamic outputs \
`stats.dat` volume and enthalpy data from every 100 steps 

This data can be analysed directly and visualised with tools such as **ASE**, **MDAnalysis**, **NumPy**, **Ovito** and **Matplotlib** .

### 5. Analysis

Post-processing includes:

- Density plot
- Volume fluctuation analysis
- Compressability / Bulk Modulus  
- Specific Heat Capacity
- Viscosity

## Polymer Construction

In [ ]:
from swiftpol import build
from openff.toolkit import Molecule, Topology, unit
from openff.interchange import Interchange
from openff.toolkit.typing.engines.smirnoff import ForceField
import random
from rdkit import Chem
import subprocess
import numpy as np

In [ ]:
# Swiftpol used to build a system of 10 PDMS chains of 40 monomers.
sys = build.polymer_system_from_PDI(
            monomer_list=['I-[Si](-C)(-C)-O-I'],
            reaction='[O:1][I:3].[Si:2][I:4]>>[O:1][Si:2].[I:3][I:4]',
            length_target=10,
            terminals='C',
            num_chains=5,
            PDI_target=1.0,
            acceptance=5
        )

# Small fix to replace -OH terminal group with SiMe3 group (-OH terminals not yet paramaterized in the bespoke force field) 
def terminal_fix(sys):
    # rdmol_chain = sys.chain_rdkit[0]
    for x in range(len(sys.chain_rdkit)):
        rdmol_chain = sys.chain_rdkit[x]
        # initialize new terminal group
        terminal = Chem.MolFromSmiles("[Si](-C)(-C)(-C)")
        terminal = Chem.AddHs(terminal)
        rdmol_chain = Chem.RemoveAllHs(rdmol_chain)
        #Copy over old atom information to new terminal group (important if you're using polyply down the line)
        index_old_atom = rdmol_chain.GetSubstructMatch(Chem.MolFromSmarts("[Si](-[OH])(-C)(-C)"))[0]
        old_atom = rdmol_chain.GetAtomWithIdx(index_old_atom)
        info = old_atom.GetPDBResidueInfo()
        [atom.SetMonomerInfo(info) for atom in terminal.GetAtoms()]
        # replace old terminal group with new one
        rdmol_chain = Chem.ReplaceSubstructs(rdmol_chain, Chem.MolFromSmarts("[Si](-[OH])(-C)(-C)"), terminal)[0]
        Chem.SanitizeMol(rdmol_chain)
        rdmol_chain = Chem.AddHs(rdmol_chain)
        for atom in rdmol_chain.GetAtoms():
            info = atom.GetPDBResidueInfo()
            if info is None:
                bonded_atom = atom.GetNeighbors()[0]
                info = bonded_atom.GetPDBResidueInfo()
                atom.SetMonomerInfo(info)
        sys.chain_rdkit[x] = rdmol_chain

    return sys

polymer = terminal_fix(sys)
polymer



In [ ]:
# Randomly assign coordinates to the polymer chains needed for polyply gen_coords
def generate_random_coordinates(mol):
    """
    Assign random 3D coordinates to all atoms in the molecule.
    """
    num_atoms = mol.GetNumAtoms()
    conf = Chem.Conformer(num_atoms)
    for i in range(num_atoms):
        # Generate random x, y, z coordinates in a reasonable range
        x, y, z = random.uniform(-10, 10), random.uniform(-10, 10), random.uniform(-10, 10)
        conf.SetAtomPosition(i, (x, y, z))
    mol.RemoveAllConformers()  # Clear existing conformers
    mol.AddConformer(conf, assignId=True)

# Assign random coordinates
for mol in polymer.chain_rdkit:
    generate_random_coordinates(mol)

# Turns the rdkit molecules into openff molecules for use with the openff toolkit
polymer.chains = [Molecule.from_rdkit(m) for m in polymer.chain_rdkit] # Includes residual monomer and oligomer

# Generate unique atom names for each molecule in the system
for molecule in polymer.chains:
    molecule.generate_unique_atom_names()


In [ ]:
# Parameterize with OpenFF

FFpath = "../bespoke_ff.offxml"

topology = Topology.from_molecules(polymer.chains)
interchange = Interchange.from_smirnoff(topology = topology, 
                                        force_field=ForceField(FFpath)
                                        )

In [ ]:
# Checks the total charge of the first chain and shows the parameters used for each atom in the first chain

charge_dict = interchange["Electrostatics"].charges
keys = sorted(charge_dict, key=lambda k: k.atom_indices[0])

# Total charge of one chain (should be 0 for a neutral system)
molecules = list(topology.molecules)
n_atoms = molecules[0].n_atoms
charges = [charge_dict[k].m for k in keys]
total_charge = sum(charges[:n_atoms])
print(f"First chain total charge = {total_charge:.6f} e")


charge_by_atom = {
    key.atom_indices[0]: charge.m
    for key, charge in charge_dict.items()}

# LibraryCharge matches
ff = ForceField(FFpath)
library_handler = ff["LibraryCharges"]
matches = library_handler.find_matches(topology)

print(f"{'AtomID':>6} {'Element':>4} {'Charge':>10} {'Library Charge'}")

all_atoms = list(topology.atoms)

# Loop over atoms in the topology
for atom_index in range(n_atoms):
    
    atom = all_atoms[atom_index]
    charge = charge_by_atom.get(atom_index, None)
    charge_id = "NOT_FOUND"

    # Find the LibraryCharge parameter applied to this atom
    for match, parameter in matches.items():
        if atom_index in match:
            charge_id = parameter.parameter_type.id
            break
    print(
        f"{atom_index:6d} "
        f"{atom.symbol:>4s} "
        f"{charge:10.4f} "
        f"{charge_id}"
    )

## Box Packing

In [ ]:
# export to .top file for polyply
interchange.to_top('polymer.top')

# Generate coordinates with polyply (overides the random coordinates)
subprocess.run([
    'polyply', 'gen_coords',
    '-p', 'polymer.top',
    '-name', 'test',
    '-dens', '950',
    '-o', 'polymer.gro'], 
    check=True
)

In [ ]:
from ase.io import read, write
from aseMolec import anaAtoms as aa

atoms = read("polymer.gro")

topology.set_positions(
    atoms.positions * unit.angstrom
)

interchange = ff.create_interchange(topology)

interchange.box = atoms.cell.lengths() * unit.angstrom

interchange.visualize()


In [ ]:
### Checks if polyply gen_coords worked. If this outputs overalpping atoms rerun the polyply gen_coords
coords = {}

with open("polymer.gro") as f:
    for line in f.readlines()[2:]:  # skip header

        parts = line.split()

        if len(parts) < 6:  # skip box line
            continue

        coord = tuple(parts[3:6])

        if coord in coords:
            print(
                f"Duplicate coordinates: atom {coords[coord]} and atom {parts[2]} at {coord}"
            )
        else:
            coords[coord] = parts[2]

## LAMMPS

This section covers the **energy minimisation**, **heating**, **cooling**, and **production molecular dynamics** stages of the simulation workflow.
LAMMPS is the molecular dynamics engine used here, although it can be replaced with alternatives such as **GROMACS** or **OpenMM** if desired.

#### Energy Minimisation
The initial configuration generated by Polyply first undergoes energy minimisation. This reduces large forces and stabilises the system before molecular dynamics begins.

#### Heating and Equilibration
To ensure the polymer chains are able to relax and sample a wide range of conformation the system is heated.

The following protocol is used under the **NPT ensemble**:

1. **0.1 ns heating**
   - Temperature ramped from **300 K → 600 K**
   - Pressure maintained at **1 atm**

2. **0.9 ns equilibration**
   - Temperature held at **600 K**
   - Pressure maintained at **1 atm**

#### Cooling and Equilibration
Following high-temperature equilibration, the system is cooled back to the target simulation temperature.

1. **2 ns cooling**
   - Temperature ramped from **600 K → 300 K**
   - Pressure maintained at **1 atm**

2. **2 ns equilibration**
   - Temperature held at **300 K**
   - Pressure maintained at **1 atm**

This stage allows the polymer density and structure to relax to equilibrium conditions prior to production runs.

#### Production NPT Simulation

After equilibration, a production simulation is performed in the **NPT ensemble**:

- Temperature: **300 K**
- Pressure: **1 atm**
- Timestep: **1 fs**
- Simulation length: **10,000,000 steps**
- Total simulation time: **10 ns**

During production:

- Thermodynamic properties are written every **1000 timesteps**
- Trajectory coordinates are written every **5000 timesteps**
- Volume and enthalpy fluctuations are written every **100 timesteps**

In [ ]:
# File system
from pathlib import Path

def stage_files(stage):
    stage_dir = Path(stage)
    stage_dir.mkdir(exist_ok=True)

    return {
        "dir": stage_dir,
        "input": stage_dir / f"lammps_openFF_MD_{stage}.in",
        "traj": f"openFF_MD_{stage}.lammpstrj",
        "log": f"openFF_MD_{stage}.thermo",
    }

# Initial datafile 
data_file = f"polymer.data"
interchange.to_lammps_datafile(f"{data_file}")

reference_masses = {
            "O": 15.99943,
            "Si": 28.08553,
            "C": 12.01078,
            "H": 1.007947,
        }

with open(f"{data_file}") as f:
    lines = f.readlines()

start = lines.index("Masses\n")
masses = []
for line in lines[start+2:]:
    if line.strip() == "":
        break
    print(line.strip())
    masses.append(float(line.strip()[2:]))

elements = []

for mass in masses:
    closest = min(
        reference_masses,
        key=lambda el: abs(reference_masses[el] - mass)
    )
    elements.append(closest)

elements = " ".join(elements)
print(elements)

In [ ]:
# Set up LAMMPS input for energy minimisation

def setup_lammps_files_min(
    MD_output_file: str,
    data_filename: str,
    input_filename: str,
    logfile: str,
    elements: list,
    thermo_frequency: int = 1000,
):

    data_file = data_filename
    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes

bond_style harmonic
angle_style harmonic
dihedral_style fourier 

read_data {data_file}

thermo_style custom step pe ebond eangle edihed etotal density press temp vol

kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

thermo {thermo_frequency}

fix relaxbox all box/relax iso 1.0 vmax 0.001

min_style cg
min_modify dmax 0.1

minimize 1.0e-6 1.0e-8 10000 100000

unfix relaxbox
write_data minimised.data
""")

files = stage_files("Min")

setup_lammps_files_min(
    MD_output_file=files["traj"], 
    data_filename=f"../{data_file}", 
    input_filename=files["input"],
    logfile=files["log"],
    elements=elements, 
) 

In [ ]:
# Run the minimization in LAMMPS
## OMP_NUM_THREADS=8 uses 8 thread if your lammps installation dosent support openMP try "!cd Min && mpirun -np 8 lmp -in lammps_openFF_MD_Min.in"
## or just use default "!cd Min && lmp -in lammps_openFF_MD_Min.in"

!cd Min && OMP_NUM_THREADS=8 lmp -sf omp -pk omp 8 -in lammps_openFF_MD_Min.in

In [ ]:
def setup_lammps_files_heat(
    MD_output_file: str,
    input_filename: str,
    data_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,   # fs
    press: float = 1.0,      # atm
    temp: float = 300,
    target_temp: float = 600,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):

    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

# Non-bonded interactions
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes # 'tail yes' adds long-range corrections to energy and pressure for truncated LJ interactions

# Bonded interactions
bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_data {data_filename}
reset_timestep 0

thermo_style custom step pe ebond eangle edihed etotal density press temp vol
thermo_modify format float %14.6f
thermo {thermo_frequency}

# long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

velocity all create {temp} 12345 mom yes rot yes dist gaussian

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}



fix 1 all npt temp {temp} {target_temp} 100.0 iso {press} {press} 1000.0
run 200000

unfix 1

fix 1 all npt temp {target_temp} {target_temp} 100.0 iso {press} {press} 1000.0
run 1800000 

unfix 1

write_data heat.data
write_restart heat.restart # writes as .restart to move to next stage of simulation

""")

files = stage_files("Heat")

setup_lammps_files_heat(
    MD_output_file=files["traj"],
    data_filename="../Min/minimised.data",
    input_filename=files["input"],
    logfile=files["log"],
    elements=elements,
)


In [ ]:
# Run the heating simulation in LAMMPS
!cd Heat && OMP_NUM_THREADS=8 lmp -sf omp -pk omp 8 -in lammps_openFF_MD_Heat.in

In [ ]:
def setup_lammps_files_cool (
    MD_output_file: str,
    input_filename: str,
    data_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,
    press: float = 1.0,      # atm
    temp: float = 300,
    target_temp: float = 600,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):

    input_file = input_filename

    with open(input_file, "w") as f:

        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

# Non-bonded interactions
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes # 'tail yes' adds long-range corrections to energy and pressure for truncated LJ interactions

# Bonded interactions
bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_restart {data_filename}
reset_timestep 0

thermo_style custom step pe ebond eangle edihed etotal density press temp vol
thermo_modify format float %14.6f
thermo {thermo_frequency}

# long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

fix 1 all npt temp {target_temp} {temp} 100.0 iso {press} {press} 1000.0
run 2000000 # 2ns cooling npt from 600 K to 300 K

unfix 1

fix 1 all npt temp {temp} {temp} 100.0 iso {press} {press} 1000.0

run 2000000 # 2ns npt at 300 K and 1 atm

unfix 1

write_data cool_eq.data
write_restart cool_eq.restart


""")
        
files = stage_files("Cool")

setup_lammps_files_cool(
    MD_output_file=files["traj"],
    data_filename="../Heat/heat.restart",
    input_filename=files["input"],
    logfile=files["log"],
    elements=elements,
)        


In [ ]:
# Run the cooling simulation in LAMMPS
!cd Cool && OMP_NUM_THREADS=8 lmp -sf omp -pk omp 8 -in lammps_openFF_MD_Cool.in

In [ ]:
def setup_lammps_files_prod(
    MD_output_file: str,
    input_filename: str,
    data_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,
    temp: float = 300.0,
    press: float = 1.0,
    n_steps = 20000000,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):

    with open(input_filename, "w") as f:
        f.write(f"""
units real
atom_style full
boundary p p p
                
log {logfile}

# Non-bonded interactions
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes # 'tail yes' adds long-range corrections to energy and pressure for truncated LJ interactions

# Bonded interactions
bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_restart {data_filename}
reset_timestep 0

thermo_style custom step pe ebond eangle edihed etotal density press temp vol
thermo_modify format float %14.6f
thermo {thermo_frequency}

# long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}


fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

fix 1 all npt temp {temp} {temp} 100.0 iso {press} {press} 1000.0

# Collect volume + enthalpy statistics for calculating compressibility and Heat capacity
variable V equal vol
variable V2 equal vol*vol
variable H equal enthalpy
variable H2 equal enthalpy*enthalpy

fix volstats all ave/time 1 1 100 v_V v_V2 v_H v_H2 file stats.dat
 
run {n_steps}

unfix 1
unfix volstats

write_data production_final.data
write_restart production_final.restart
""")

files = stage_files("Prod")

setup_lammps_files_prod(
    MD_output_file=files["traj"],
    data_filename="../Cool/cool_eq.restart",
    input_filename=files["input"],
    logfile=files["log"],
    elements=elements,
)

In [ ]:
!cd Prod && OMP_NUM_THREADS=8 lmp -sf omp -pk omp 8 -in lammps_openFF_MD_Prod.in

## Testing 

Different production runs need to be used for calculating different properties

Density, Heat Capacity, Compressability are all done in Equlibrium MD with NPT over 10 ns

Viscosity is done in Equlibrium MD NVT for 10 ns

In [ ]:
def setup_lammps_files_prod_NVT(
    MD_output_file: str,
    input_filename: str,
    data_filename: str,
    logfile: str,
    elements: list,
    timestep: float = 0.5,
    temp: float = 300.0,
    press: float = 1.0,
    xyz_frequency: int = 5000,
    thermo_frequency: int = 1000,
):
    n_steps = 20000000
    time_ps = n_steps * timestep / 1000
    time_ns = n_steps * timestep / 1000000
    print(f"Simulation length: {time_ps:.1f} ps {time_ns:.3f} ns")

    with open(input_filename, "w") as f:
        f.write(f"""
units real
atom_style full
boundary p p p

log {logfile}

# Force field styles
pair_style lj/cut/coul/long 11 11
pair_modify mix arithmetic tail yes

bond_style harmonic
angle_style harmonic
dihedral_style fourier

read_restart {data_filename}

# Long-range electrostatics
kspace_style pppm 1e-4

neighbor 2.0 bin
neigh_modify delay 0 every 1 check yes

timestep {timestep}

thermo_style custom step pe ebond eangle edihed etotal density press temp vol
thermo {thermo_frequency}

fix remove_drift all momentum 100 linear 1 1 1

dump traj all custom {xyz_frequency} {MD_output_file} id element xu yu zu
dump_modify traj sort id element {elements}

fix 1 all nvt temp 300 300 100.0 

variable pxx equal pxx
variable pyy equal pyy
variable pzz equal pzz
variable pxy equal pxy
variable pxz equal pxz
variable pyz equal pyz
variable step equal step

fix stress all print 10 "$(step) $(pxx) $(pyy) $(pzz) $(pxy) $(pxz) $(pyz)" file stress.dat screen no title "step pxx pyy pzz pxy pxz pyz"
run {n_steps}

unfix 1

write_data NVT.data
""")
        
files = stage_files("Prod_NVT")

setup_lammps_files_prod_NVT(
    MD_output_file=files["traj"],
    data_filename="../Cool/cool_eq.restart",
    input_filename=files["input"],
    logfile=files["log"],
    elements=elements,
)

In [ ]:
!cd Prod_NVT && OMP_NUM_THREADS=8 lmp -sf omp -pk omp 8 -in lammps_openFF_MD_Prod_NVT.in

## Analysis

This section demonstrates how to analyse data generated during MD and extract key thermodynamic properties of PDMS.

### Output Files
- **`.lammpstrj`**
Trajectory file containing atomic coordinates (XYZ data), written every **2000 timesteps**.
 
- **`.thermo`**
Thermodynamic output containing quantities such as:
    - Step
    - Potential Energy (`pe`)
    - Bond Energy (`ebond`)
    - Angle Energy (`eangle`)
    - Dihedral Energy (`edihed`)
    - Total Energy (`etotal`)
    - Density (`density`)
    - Pressure (`press`)
    - Temperature (`temp`)
    - Volume (`vol`)

- **`stats.dat`** 
Contains **volume** and **enthalpy** statistics used to calculate:
    - Compressibility
    - Specific heat capacity




### Notes on Fluctuation Properties
`stats.dat` contains **instantaneous** values for volume and enthapy recorded every **100 timesteps**. This is to avoid infomation loss  that occurs when averaging prior to analysis. Averaging can supress the true variance of volume leading to underestimation of compressability which is dependant on the fluctuations. 

In [ ]:
total_atoms = topology.n_atoms
print(f"Total atoms in system = {total_atoms}")

In [ ]:
## pandas used to make analysis easier from header names

import pandas as pd

with open("Prod/openFF_MD_Prod.thermo") as f:
    lines = f.readlines()

# Find thermo header
for i, line in enumerate(lines):
    if line.strip().startswith("Step"):
        header = line.split()
        start_line = i + 1
        break

rows = []

for line in lines[start_line:]:
    parts = line.split()

    # Skip anything that doesn't match thermo column count
    if len(parts) != len(header):
        continue

    try:
        rows.append([float(x) for x in parts])
    except ValueError:
        continue

thermopd = pd.DataFrame(rows, columns=header)

### Volume

In [ ]:
# Volume histogram

import matplotlib.pyplot as plt
data = np.loadtxt("Prod/stats.dat", comments="#")

V = data[:,1]   # <V> values

plt.hist(V, bins=50)
plt.xlabel("Volume (Å³)")
plt.ylabel("Count")

plt.savefig("volume_histogram.png", dpi=300)
plt.show()

In [ ]:
# Calculate skewness of the volume distribution
from scipy.stats import skew
print(skew(V))

Skew > 0 right tailed \
Skew < 0 left tailed \
Skew = 0 no tail \ 

Skew = 0.095 neglegable difference system likely in equilibrium 


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thermo = np.loadtxt("Prod/stats.dat", comments="#")

# Extract columns
step = thermo[:, 0]
Volume = thermo[:, 1]

window = 10000 

rolling_Volume = np.convolve(
    Volume,
    np.ones(window)/window,
    mode="valid"
)

rolling_step = step[window-1:]

plt.figure(figsize=(8,5))
plt.plot(step, Volume, alpha=0.4, color='steelblue', label='Raw data')
plt.plot(rolling_step, rolling_Volume,
         linewidth=2, label=f'Rolling average')

plt.xlabel("Timestep (1 fs)")
plt.ylabel("Volume (A³)")
plt.title("Volume vs Timestep")
plt.legend()
plt.tight_layout()
plt.show()

### Compressibility

In [ ]:
### Calculate compressibility from volume statistics

import numpy as np

data = np.loadtxt("Prod/stats.dat")
V = data[:, 1]  # Extract the volume column
V2 = data[:, 2]  # Extract the volume squared column

mean_V = np.mean(V)
mean_V2 = np.mean(V2)

varV = mean_V2 - mean_V**2

kB = 1.380649e-23  # Boltzmann constant in J/K
T = 300  # Temperature in K

mean_V *= 1e-30 # Å^3 -> m^3
varV *= 1e-60 # Å^6 -> m^6

compressibility = varV/(kB*T*mean_V)  # Calculate compressibility
print(f"Compressibility: {compressibility} Pa^-1")
print(f"Bulk Modulus: {1/compressibility/1e9} GPa")

### Specific Heat Capacity

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thermo = np.loadtxt("Prod/stats.dat", comments="#")

# Extract columns
step = thermo[:, 0]
Enthalpy = thermo[:, 3]

window = 10000 

rolling_enthalpy = np.convolve(
    Enthalpy,
    np.ones(window)/window,
    mode="valid"
)

rolling_step = step[window-1:]

plt.figure(figsize=(8,5))
plt.plot(step, Enthalpy, alpha=0.4, color='steelblue', label='Raw data')
plt.plot(rolling_step, rolling_enthalpy,
         linewidth=2, label=f'Rolling average')

plt.xlabel("Timestep (1 fs)")
plt.ylabel("Enthalpy (kcal/mol)")
plt.title("Enthalpy vs Timestep")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
### Calculate Specific Heat Capacity from enthalpy in stats.dat

import numpy as np 

data = np.loadtxt("Prod/stats.dat")
Hstat = data[:, 3]  # Extract the enthalpy column
H2 = data[:, 4]  # Extract the enthalpy squared column (kcal/mol)^2

mean_H = np.mean(Hstat)   # kcal/mol
mean_H2 = np.mean(H2) # (kcal/mol)^2

# varH = mean_H2 - mean_H**2

varH = np.var(Hstat[::10])  # Variance of enthalpy

total_mass = sum(atom.mass for atom in topology.atoms).m # Da = g/mol
print(total_mass)
R = 1.987204259e-3  # Gas constant in kcal/(mol*K)
kB = 1.380649e-23  # Boltzmann constant in J/K
T = 300  # Temperature in K 

specific_heat = varH/(R*T**2*total_mass)  # Calculate specific heat capacity
print(f"Specific Heat Capacity: {specific_heat} kcal/(g*K)")
print(f"Specific Heat Capacity: {specific_heat*4184} J/(g*K)")

Back up check for Specific Heat Capacity from the thermo file

enthalpy = etotal + press*vol <- What Lammps calculates for enthalpy \
H = U + PV


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

TotEng = thermopd["TotEng"] #kcal/mol
PotEng = thermopd["PotEng"] #kcal/mol
Press = thermopd["Press"] #atm
Vol = thermopd["Volume"] #A^3

PV = Press * Vol * 1.4583975e-5
H = PotEng + PV

mean_H = np.mean(H)
mean_H2 = np.mean(H**2)
#varH = mean_H2 - mean_H**2
varH = np.var(H)  # Variance of enthalpyvarH = mean_H2 - mean_H**2

total_mass = sum(atom.mass for atom in topology.atoms).m # Da = g/mol
print(total_mass)
R = 1.987204259e-3  # Gas constant in kcal/(mol*K)
kB = 1.380649e-23  # Boltzmann constant in J/K
T = 300  # Temperature in K

specific_heat = varH/(R*T**2*total_mass)  # Calculate specific heat capacity
print(f"Specific Heat Capacity: {specific_heat} kcal/(g*K)")
print(f"Specific Heat Capacity: {specific_heat*4184} J/(g*K)")

### Density

In [ ]:
import pandas as pd


density = thermopd["Density"]
step = thermopd["Step"]

print(f"Average density = {density.mean():.4f} g/cm3")

In [ ]:
window = 1000 

rolling_density = density.rolling(window=window).mean()


plt.figure(figsize=(8,5))
plt.plot(step, density, alpha=0.4, color='steelblue', label='Raw data')
plt.plot(step, rolling_density,
         linewidth=2, label=f'Rolling average')

plt.xlabel("Timestep")
plt.ylabel("Density (g/cm³)")
plt.title("Density vs Timestep")
plt.legend()
plt.tight_layout()

# plt.savefig("DensityTimeplot.png", dpi=600)
plt.show()

### Visualisation

In [ ]:
## To view the final unwraped structure use ase and aseMolec then visualise in ovito 
from ase.io import read, write
from aseMolec import anaAtoms as aa

atoms = read('../Prod/openFF_MD_Prod.lammpstrj', '::100')
aa.wrap_molecs(atoms)

write('test.xyz', atoms)

### Viscosity

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

with open("Prod_NVT/openFF_MD_Prod_NVT.thermo") as f:
    lines = f.readlines()

# Find thermo header
for i, line in enumerate(lines):
    if line.strip().startswith("Step"):
        header = line.split()
        start_line = i + 1
        break

rows = []

for line in lines[start_line:]:
    parts = line.split()

    # Skip anything that doesn't match thermo column count
    if len(parts) != len(header):
        continue

    try:
        rows.append([float(x) for x in parts])
    except ValueError:
        continue

thermopdnvt = pd.DataFrame(rows, columns=header)

In [ ]:
def acf_short(signal, maxlag):
    signal = signal - signal.mean()
    N = len(signal)

    C = np.empty(maxlag + 1)

    for lag in range(maxlag + 1):
        C[lag] = np.dot(signal[:N-lag], signal[lag:]) / (N - lag)

    return C
df = pd.read_csv("Prod_NVT/stress.dat", sep=r"\s+")

# Conversion factor atm -> Pa 
ATM_TO_PA = 101325.0

pxy = df["pxy"].values * ATM_TO_PA
pxz = df["pxz"].values * ATM_TO_PA
pyz = df["pyz"].values * ATM_TO_PA

Cxy = acf_short(pxy, 10000)
Cxz = acf_short(pxz, 10000)
Cyz = acf_short(pyz, 10000)

print(np.mean(Cxy[300:500]))
print(np.mean(Cxz[300:500]))
print(np.mean(Cyz[300:500]))

# When starting the NVT from the NPT there will be a difference in pressue along each axis, so we need to remove the mean of the ACFs when converged to account for this.
Cxy -= np.mean(Cxy[300:500])
Cxz -= np.mean(Cxz[300:500])
Cyz -= np.mean(Cyz[300:500]) 

plt.figure(figsize=(20, 5))
plt.plot(Cxy[:500])
plt.plot(Cxz[:500])
plt.plot(Cyz[:500]) 


In [ ]:
def running_viscosity(C, 
                      Vol_A3=None, 
                      temp=300, 
                      timestep_fs=0.5, 
                      sample_every=10):
    if Vol_A3 is None:
        Vol_A3=thermopdnvt["Volume"].mean() # A^3
    
    V = Vol_A3 * 1e-30 # convert to m^3
    dt_s = timestep_fs * sample_every * 1e-15 # convert to seconds

    integral = np.cumsum(C) * dt_s
    eta = (V / (kB * temp)) * integral

    return eta


kB = 1.380649e-23

timestep_fs = 0.5
sample_every = 10

eta1 = running_viscosity(
    Cyz[:500],
    timestep_fs=timestep_fs,
    sample_every=sample_every
)

eta2 = running_viscosity(
    Cxz[:500],
    timestep_fs=timestep_fs,
    sample_every=sample_every
)

eta3 = running_viscosity(
    Cxy[:500],
    timestep_fs=timestep_fs,
    sample_every=sample_every
)

# plotting
plt.figure(figsize=(8,5))
plt.plot(lag_ps, eta1, label="Cyz")
plt.plot(lag_ps, eta2, label="Cxz")
plt.plot(lag_ps, eta3, label="Cxy")

plt.xlabel("Time (ps)")
plt.ylabel("Viscosity (Pa·s)")
plt.title(f"Running Green-Kubo Viscosity (2.5 ps)")

plt.show()



In [ ]:
eta_avg = (eta1 + eta2 + eta3)/3

plateau = np.mean(eta_avg[300:500])
std = np.std(eta_avg[300:500])

print(f"Viscosity = {plateau:.6e} ± {std:.6e} Pa.s")